<a href="https://colab.research.google.com/github/vivastarkey-hub/MIS2800/blob/main/afc_north_sqli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORTANT ❗
The below blocks are the lines of code to begin using sqlite3 magic.  

This is meant to be reused as needed.  To use, go to File > Save a Copy in Drive.


In [ ]:
!pip install --upgrade pandas ipython-sql prettytable==3.10.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.4 MB/s eta 0:00:00
  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.17.0
    Uninstalling prettytable-3.17.0:
      Successfully uninstalled prettytable-3.17.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is

In [ ]:
# Install the SQL extension. Remember to type exactly as shown here.
!pip install ipython-sql
print("ipython sql installed")

ipython sql installed


In [ ]:
# Load the SQL magic extension
%load_ext sql

In [ ]:
#this code imports sqlite3 and prints a statement stating it has been imported
import sqlite3
print("sqlite3 imported")

sqlite3 imported


In [ ]:
# This creates a brand-new, empty database file named [name of your database.db]
conn = sqlite3.connect('nameOfDatabase.db')
print("nameOfDatabase database created")


nameOfDatabase database created


In [ ]:
# This line is required so Colab stops acting like Python and acts more like a SQL database terminal
%sql sqlite:///nameOfDatabase.db

# ----SPECIFIC FOR PREP/IC/HW ----

You are inside the AFC North “Coach Terminal.”
Everything is done using SQL commands only.

Teams:

🟠 Bengals
🟤 Browns
🟡 Steelers
🟣 Ravens

# Part 1 Steps:
1. Create a table named "roster" (
  primary_id
  team
  player
  position

2. Fix the error in cell 2



In [ ]:
# Cell 1: create table in this cell
%%sql
CREATE TABLE roster (
  primary_id INTEGER PRIMARY KEY,
  team TEXT,
  player TEXT,
  position TEXT
);

 * sqlite:///nameOfDatabase.db
Done.


[]

In [ ]:
# Cell 2: what's missing for this syntax to work?
%%sql
INSERT INTO roster (team, player, position) VALUES ('Bengals', 'Joe Burrow', 'QB');
INSERT INTO roster (team, player, position) VALUES ('Bengals', 'Ja''Marr Chase', 'WR');

INSERT INTO roster (team, player, position) VALUES ('Browns', 'Myles Garrett', 'DE');
INSERT INTO roster (team, player, position) VALUES ('Browns', 'Deshaun Watson', 'QB');

INSERT INTO roster (team, player, position) VALUES ('Steelers', 'T.J. Watt', 'LB');
INSERT INTO roster (team, player, position) VALUES ('Steelers', 'Najee Harris', 'RB');

INSERT INTO roster (team, player, position) VALUES ('Ravens', 'Lamar Jackson', 'QB');
INSERT INTO roster (team, player, position) VALUES ('Ravens', 'Mark Andrews', 'TE');

 * sqlite:///nameOfDatabase.db
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.


[]

# Vulnerable "Coach Search" Query (SQL Injection Risk)
### Bad Practice: String-built SQL
Problem: Database is accepting raw SQL fragments disguised as input. The user input is NOT treated as data - it is treated as code inside the query.

The query will take the text --> insert directly into SQL --> execute everything. The user input becomes part of SQL syntax in the table.

This can be done with ***OR 1=1*** attack

Example:
  SELECT * FROM roster
  WHERE team = '' OR '1'='1';

SQL will interpret as
  team = '' --> False
  '1' = '1' --> always true
So overall condition is TRUE because of the "OR" (FALSE OR TRUE --> TRUE)

Every row is returned.

In [ ]:
%%sql
SELECT * FROM roster
WHERE team = ' ' OR '1'='1';

 * sqlite:///nameOfDatabase.db
Done.


primary_id,team,player,position
1,Bengals,Joe Burrow,QB
2,Bengals,Ja'Marr Chase,WR
3,Browns,Myles Garrett,DE
4,Browns,Deshaun Watson,QB
5,Steelers,T.J. Watt,LB
6,Steelers,Najee Harris,RB
7,Ravens,Lamar Jackson,QB
8,Ravens,Mark Andrews,TE


## Why the Quotes Matter
Single quotes are supposed to mean text string but attackers can break out of them.
in the example of ***' OR '1'='1***
SQL now things the ' closes the string early and that everything after becomes real SQL logic

# Parameterized Query

Parameterized Query: way of writing a db query where you separate the SQL code from the user input.

How it Works: Instead of putting user input directly into a SQL statement, you use a placeholder and then plug the value in later.

## Example of Unsafe (Not Parameterized)
SELECT * FROM users WHERE username = 'alice'

## Example of Safe (Parameterized)
SELECT * FROM users WHERE username = ?;

The ? is a placeholder that is telling the db to fill it in later with a value. So if the user enters "alice", **the db will treat it strictly as a value, not a code.**

# Why This Matters

Parameterization protects you because:


*   user input is never treated as SQL code
*   user input is treated as data only
*   helps prevent SQLi attacks



In [ ]:
# safe way to write to avoide SQLi
%%sql
SELECT * from roster
WHERE team = ?

 * sqlite:///nameOfDatabase.db
(sqlite3.ProgrammingError) Incorrect number of bindings supplied. The current statement uses 1, and there are 0 supplied.
[SQL: SELECT * from roster
WHERE team = ?]
(Background on this error at: https://sqlalche.me/e/20/f405)


While it is outside the scope of this class to go deeply into web architecture, it is helpful to understand that most modern organizations design their systems using a **three-tier architecture**.

1. **Web interface (presentation layer)** – This is what users see and interact with in a browser or app. It includes forms, buttons, search bars, and visual content.
2. **Application layer (logic layer)** – This is where the system’s programming logic runs. It processes user input, applies rules, and determines what data should be retrieved or modified.
3. **Database layer (data layer)** – This is where information is stored and managed. It is typically not directly accessible to the public and is only interacted with through the application layer.

An important security concept in this structure is that users also **do not see how database queries are actually constructed or executed**. In secure systems, the application layer uses **parameterized queries (or prepared statements)** to communicate with the database. This means the SQL structure is defined in advance, and user input is passed separately as data values rather than being directly inserted into the query itself.

As a result, even though a user may submit input through a form, they never see the underlying query logic, and they do not control how that input is interpreted by the database. This separation helps prevent issues such as SQL injection by ensuring that user input cannot alter the structure of the SQL commands being executed.


3 tier architecture. 1 web interface - what consumers interact with (graphical user interface GUI). 2 application layer - python code, java script, logic. 3 data base layer where SQL actually runs - users do not have access to this layer info is stored here.